
# QML-SleepNet — Stage 11 Final Evaluation Protocol & Statistical Comparison
## Promoted 90.8627% QML-inclusive final system — v1

### Direct roadmap contract

This notebook implements **Stage 11 — FINAL EVALUATION PROTOCOL (As per Original Paper Style)** from the supplied `QML-SleepNet Research Roadmap.png`.

**11.1 Standard Evaluation**
- use official test set;
- report class-wise metrics;
- compare with state-of-the-art / original-challenge reference results.

**11.2 Statistical Analysis**
- paired t-tests;
- Cohen's `d`;
- confidence intervals;
- significance testing.

**11.3 Comparison**
- VQC vs QSVC;
- Quantum vs Classical;
- ablation results;
- discussion of performance gains.

**11.4 Final Results**
- best-model designation;
- summary tables and plots;
- key findings and limitations.

Roadmap footer requirements:
- reproduce/reference original-paper results;
- same dataset and evaluation protocol where legitimately possible;
- report all metrics consistently;
- ensure fair comparison.

---

## Scientific locks

The model-development stage is over.

The final metric-first QML-inclusive system was already promoted from **strict development OOF evidence** before its fused x score was opened:

\[
\boxed{
P_{\rm final}
=
\sigma\!\left[
0.25\,\operatorname{logit}(P_{\rm physiology})
+
0.75\,\operatorname{logit}(P_{\rm QML})
\right]
}
\]

with threshold 0.5.

Expected promoted-final prediction SHA-256:

`063a017e61188393bcdcdacb72958ffa3e7e0efa9432d33aa0845983462dfa1f`

Expected official-x accuracy over the frozen project evaluation universe:

\[
\boxed{90.86270871985158\%}
\]

### This notebook performs

- **ZERO training**
- **ZERO fine-tuning**
- **ZERO model selection**
- **ZERO threshold search**
- **ZERO fusion-weight search**
- **ZERO calibration/HMM refit**
- **ZERO new feature engineering**

The roadmap phrase **"Best model selection"** is interpreted here as **final designation/reporting**, not test-set-driven winner shopping.

### Statistical unit

Headline classification metrics remain minute-level because Task A is minute-level apnea quantification.

However, inferential statistics do **not** pretend 17,248 serially correlated minutes are independent.

The primary paired statistical units are the **35 official x-record clusters**.  
They are called *record clusters*, not *subjects*, because this notebook does not assume one-to-one unique-subject identity beyond the frozen record contract.

### External benchmark boundary

The official PhysioNet/Computing in Cardiology Challenge 2000 Event 2 leaderboard scored **17,268** reference-annotated minutes. This project evaluates **17,248 complete observed 60-s ECG minutes** and deliberately does not fabricate/pad terminal partial minutes.

Therefore historical Challenge scores are included for **authoritative context only**, not as a protocol-identical head-to-head SOTA claim.


In [ ]:

# Cell 1 — Drive, imports, immutable paths

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, uuid, warnings, math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    confusion_matrix, brier_score_loss, log_loss,
    precision_recall_fscore_support, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("/content/drive/MyDrive/QML_SleepNet")
GX = ROOT / "outputs/GUIDE_EXACT_METRICMAX"

# Promoted final.
FUSION_ROOT = GX / "FINAL_METRIC_CHAMPION_FIXED_FUSION_AUDIT_v1"
FUSED_X = FUSION_ROOT / "FINAL_FIXED_FUSION_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"
FUSION_FREEZE = FUSION_ROOT / "FINAL_FIXED_FUSION_PRE_SCORE_FREEZE_MANIFEST.json"
FUSION_REPORT = FUSION_ROOT / "FINAL_FIXED_FUSION_POSTHOC_X_METRICS.json"
FUSION_DECISION = FUSION_ROOT / "FINAL_FIXED_FUSION_DEVELOPMENT_DECISION.json"
FUSION_BOOTSTRAP_DEV = FUSION_ROOT / "FIXED_FUSION_STRICT_GROUP_BOOTSTRAP.csv"

# Parent frozen files — used for SHA provenance; parent probabilities are also embedded in FUSED_X.
PHYS_ROOT = GX / "DEV35_STAGE15A_PHYSIO_BOOSTING_CHALLENGER_v1_2_COVERAGESAFE"
PHYS_X = PHYS_ROOT / "DEV35_STAGE15A_PHYSIO_BOOSTING_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"

QML_ROOT = GX / "FINAL_FIXED_EQUAL_LOGIT_ENSEMBLE_v1"
QML_X = QML_ROOT / "FINAL_FIXED_EQUAL_LOGIT_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"

# Raw official label sources.
RAW = ROOT / "data/raw/apnea_ecg"
LABEL_JSON = RAW / "test_set_apnea_labels.json"
LABEL_TXT = RAW / "test-dataset-annos.txt"

# Promoted-final Stage09/10 evidence.
XAI_ROOT = GX / "XAI_CAUSAL_INTERPRETABILITY_90P8627_PROMOTED_FINAL_v1"
XAI_MANIFEST = XAI_ROOT / "XAI_PROMOTED_FINAL_MANIFEST.json"

S10_ROOT = GX / "STAGE10_MODEL_ROBUSTNESS_90P8627_PROMOTED_FINAL_v1"
S10_MANIFEST = S10_ROOT / "STAGE10_PROMOTED_FINAL_MANIFEST.json"
S10_ABLATION = S10_ROOT / "STAGE10_PROMOTED_PARENT_ABLATION.csv"
S10_PERTURB = S10_ROOT / "STAGE10_PROMOTED_PERTURBATION_ROBUSTNESS.csv"
S10_DOMAIN = S10_ROOT / "STAGE10_PROMOTED_GENERALIZATION_DOMAIN_SHIFT.csv"
S10_SCOPE = S10_ROOT / "STAGE10_PROMOTED_UNSUPPORTED_OR_SCOPE_LIMITED_ITEMS.json"

# Guide-parity Stage06 VQC/QSVC development evidence.
E2_ROOT = ROOT / "outputs/performance_core/stage15e2_r3_anisotropic_angle_metricmax_v1_0"
E2_COMPARE = E2_ROOT / "stage15e2_r3_final_comparison.csv"
E2_DECISION = E2_ROOT / "STAGE15E2_R3_FINAL_DECISION.json"
E2_QSVC_FOLDS = E2_ROOT / "fixed_qsvc_fourfold_raw.csv"
E2_VQC_FOLDS = E2_ROOT / "vqc_fourfold.csv"

# New Stage11 output.
OUT = GX / "STAGE11_FINAL_EVALUATION_90P8627_v1"
PLOTS = OUT / "plots"
OUT.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(parents=True, exist_ok=True)

EXPECTED_FUSED_SHA = "063a017e61188393bcdcdacb72958ffa3e7e0efa9432d33aa0845983462dfa1f"
EXPECTED_QML_SHA = "f121a79191be00a28f33e06e7dec20cc689268b10a988c52e21284c90d1e2eef"
EXPECTED_PHYS_SHA = "e548af5c1d8ad39f8ea192680ee7ae7af303b993e65f410ea2ee18f015e3e244"
EXPECTED_ACC = 0.9086270871985158
EXPECTED_N = 17248
EXPECTED_W_PHYS = 0.25
EXPECTED_W_QML = 0.75
EXPECTED_THRESHOLD = 0.5
EPS = 1e-8
BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20260914

required = [
    FUSED_X, FUSION_FREEZE, FUSION_REPORT, FUSION_DECISION, FUSION_BOOTSTRAP_DEV,
    PHYS_X, QML_X, LABEL_JSON, LABEL_TXT,
    XAI_MANIFEST, S10_MANIFEST, S10_ABLATION, S10_PERTURB, S10_DOMAIN, S10_SCOPE,
    E2_COMPARE, E2_DECISION, E2_QSVC_FOLDS, E2_VQC_FOLDS,
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required frozen Stage11 input(s):\n" + "\n".join(missing))

print("Stage 11 initialized.")
print("ZERO TRAINING / ZERO TUNING / ZERO MODEL SELECTION")
print("Output:", OUT)


In [ ]:

# Cell 2 — deterministic helpers

def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_name(path.name + f".tmp.{uuid.uuid4().hex}")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str))
    os.replace(tmp, path)

def atomic_csv(path, df):
    path = Path(path)
    tmp = path.with_name(path.name + f".tmp.{uuid.uuid4().hex}")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def atomic_savefig(path, fig):
    path = Path(path)
    tmp = path.with_name(path.stem + f".tmp.{uuid.uuid4().hex}" + path.suffix)
    fig.savefig(tmp, dpi=180, bbox_inches="tight")
    os.replace(tmp, path)

def logit(p):
    p = np.clip(np.asarray(p, np.float64), EPS, 1-EPS)
    return np.log(p) - np.log1p(-p)

def sigmoid(z):
    z = np.asarray(z, np.float64)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0/(1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez/(1.0 + ez)
    return out

def fixed_fusion(p_phys, p_qml, w_phys=EXPECTED_W_PHYS):
    return sigmoid(float(w_phys)*logit(p_phys) + (1.0-float(w_phys))*logit(p_qml))

def metric_row(y, p, threshold=0.5):
    y = np.asarray(y, np.int8)
    p = np.asarray(p, np.float64)
    pred = (p >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    two = len(np.unique(y)) == 2
    return {
        "n": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)) if two else np.nan,
        "precision": float(precision_score(y, pred, zero_division=0)),
        "sensitivity": float(recall_score(y, pred, zero_division=0)),
        "specificity": float(tn/max(tn+fp,1)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y, pred)) if two else np.nan,
        "auroc": float(roc_auc_score(y, p)) if two else np.nan,
        "auprc": float(average_precision_score(y, p)) if two else np.nan,
        "brier": float(brier_score_loss(y, p)),
        "nll": float(log_loss(y, np.column_stack([1-p,p]), labels=[0,1])),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def parse_record(uid):
    return str(uid).rsplit(":", 1)[0]

def normalize_json_labels(obj):
    out = {}
    for rec, val in obj.items():
        seq = val.strip() if isinstance(val, str) else "".join(map(str, val))
        if set(seq) - {"A","N"}:
            raise RuntimeError(f"{rec}: unexpected label symbol")
        out[str(rec)] = seq
    return out

def parse_txt_labels(path):
    out = {}
    current = None
    chunks = []
    for raw in Path(path).read_text().splitlines():
        line = raw.strip()
        if not line:
            continue
        if len(line) == 3 and line.startswith("x") and line[1:].isdigit():
            if current is not None:
                out[current] = "".join(chunks)
            current = line
            chunks = []
            continue
        if current is None:
            continue
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            seq = "".join(parts[1:]).strip()
            if set(seq) - {"A","N"}:
                raise RuntimeError(f"Unexpected official annotation row: {line}")
            chunks.append(seq)
    if current is not None:
        out[current] = "".join(chunks)
    return out

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj_sorted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        raw_adj = (m-rank)*p[idx]
        running = max(running, raw_adj)
        adj_sorted[rank] = min(1.0, running)
    out = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        out[idx] = adj_sorted[rank]
    return out

def paired_stats(a, b, label_a, label_b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    d = a - b
    n = len(d)
    mean = float(np.mean(d))
    sd = float(np.std(d, ddof=1)) if n > 1 else np.nan
    dz = float(mean/sd) if np.isfinite(sd) and sd > 0 else np.nan
    test = stats.ttest_rel(a, b, nan_policy="omit")
    sem = stats.sem(d, nan_policy="omit")
    if np.isfinite(sem) and n > 1:
        ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=sem)
    else:
        ci = (np.nan, np.nan)
    return {
        "system_a": label_a,
        "system_b": label_b,
        "n_pairs": int(n),
        "mean_accuracy_a": float(np.mean(a)),
        "mean_accuracy_b": float(np.mean(b)),
        "mean_paired_difference": mean,
        "mean_paired_difference_pp": 100*mean,
        "t_statistic": float(test.statistic),
        "p_value_raw": float(test.pvalue),
        "cohen_dz": dz,
        "ci95_mean_difference_low": float(ci[0]),
        "ci95_mean_difference_high": float(ci[1]),
        "ci95_mean_difference_low_pp": 100*float(ci[0]),
        "ci95_mean_difference_high_pp": 100*float(ci[1]),
    }

print("Helpers ready.")


In [ ]:

# Cell 3 — fail-closed provenance + direct official-label reconstruction

freeze = json.loads(FUSION_FREEZE.read_text())
saved_report = json.loads(FUSION_REPORT.read_text())
xai_manifest = json.loads(XAI_MANIFEST.read_text())
s10_manifest = json.loads(S10_MANIFEST.read_text())
e2_decision = json.loads(E2_DECISION.read_text())

if sha256_file(FUSED_X) != EXPECTED_FUSED_SHA:
    raise RuntimeError("Promoted final prediction SHA mismatch")
if freeze.get("fused_prediction_sha256") != EXPECTED_FUSED_SHA:
    raise RuntimeError("Promoted freeze-manifest SHA mismatch")
if sha256_file(QML_X) != EXPECTED_QML_SHA:
    raise RuntimeError("Canonical QML parent SHA mismatch")
if sha256_file(PHYS_X) != EXPECTED_PHYS_SHA:
    raise RuntimeError("Stage15A physiology parent SHA mismatch")
if freeze.get("qml_component_x_sha256") != EXPECTED_QML_SHA:
    raise RuntimeError("Freeze manifest does not bind expected QML parent")
if freeze.get("stage15a_component_x_sha256") != EXPECTED_PHYS_SHA:
    raise RuntimeError("Freeze manifest does not bind expected physiology parent")
if freeze.get("official_x_labels_loaded") is not False:
    raise RuntimeError("Fusion was not label-sealed at freeze")
if freeze.get("training_performed") is not False:
    raise RuntimeError("Fusion audit trained unexpectedly")
if freeze.get("threshold_search") is not False:
    raise RuntimeError("Fusion threshold search detected")
if freeze.get("calibration_refit") is not False or freeze.get("hmm_refit") is not False:
    raise RuntimeError("Fusion calibration/HMM refit detected")
if freeze["winning_fusion"]["candidate"] != "FIXED_LOGIT_PHYS_25_QML_75":
    raise RuntimeError("Unexpected promoted fusion identity")
if freeze["winning_fusion"].get("promotion_gate_pass") is not True:
    raise RuntimeError("Promoted fusion did not pass development gate")
if abs(float(freeze["winning_fusion"]["w_phys"])-EXPECTED_W_PHYS) > 1e-15:
    raise RuntimeError("Physiology weight drift")
if abs(float(freeze["winning_fusion"]["w_qml"])-EXPECTED_W_QML) > 1e-15:
    raise RuntimeError("QML weight drift")

if xai_manifest.get("promoted_final_prediction_sha256") != EXPECTED_FUSED_SHA:
    raise RuntimeError("Stage09 promoted XAI SHA mismatch")
if s10_manifest.get("primary_frozen_prediction_sha256") != EXPECTED_FUSED_SHA:
    raise RuntimeError("Stage10 promoted robustness SHA mismatch")
if s10_manifest.get("training_performed") is not False:
    raise RuntimeError("Stage10 manifest says training occurred")
if s10_manifest.get("model_selection_performed") is not False:
    raise RuntimeError("Stage10 manifest says model selection occurred")

z = np.load(FUSED_X, allow_pickle=False)
required_keys = {
    "test_uids", "physiology_hmm_posterior", "qml_core3_hmm_logit_mean",
    "physiology_weight", "qml_weight", "fused_probability", "prediction",
    "hard_threshold"
}
if not required_keys.issubset(z.files):
    raise RuntimeError(f"Promoted NPZ missing keys: {required_keys-set(z.files)}")

UID = np.asarray(z["test_uids"]).astype(str)
P_PHYS = np.asarray(z["physiology_hmm_posterior"], np.float64)
P_QML = np.asarray(z["qml_core3_hmm_logit_mean"], np.float64)
P_FINAL = np.asarray(z["fused_probability"], np.float64)
PRED_FINAL = np.asarray(z["prediction"], np.int8)
W_PHYS = float(np.asarray(z["physiology_weight"]).item())
W_QML = float(np.asarray(z["qml_weight"]).item())
THR = float(np.asarray(z["hard_threshold"]).item())

if len(UID) != EXPECTED_N:
    raise RuntimeError(f"Expected {EXPECTED_N} final rows, got {len(UID)}")
if len(np.unique(UID)) != len(UID):
    raise RuntimeError("Duplicate UID in promoted final artifact")
if not (np.isfinite(P_PHYS).all() and np.isfinite(P_QML).all() and np.isfinite(P_FINAL).all()):
    raise RuntimeError("Non-finite score detected")
if abs(W_PHYS-EXPECTED_W_PHYS) > 1e-15 or abs(W_QML-EXPECTED_W_QML) > 1e-15:
    raise RuntimeError("Stored promoted weights drift")
if abs(THR-EXPECTED_THRESHOLD) > 1e-15:
    raise RuntimeError("Stored final threshold drift")

reconstructed = fixed_fusion(P_PHYS, P_QML, W_PHYS)
RECON_MAX = float(np.max(np.abs(reconstructed-P_FINAL)))
if RECON_MAX > 1e-12:
    raise RuntimeError(f"Promoted fusion reconstruction failed: {RECON_MAX}")
if not np.array_equal(PRED_FINAL, (P_FINAL >= THR).astype(np.int8)):
    raise RuntimeError("Final stored predictions disagree with score/threshold")

# Open official labels directly from both project-held official sources.
lab_json = normalize_json_labels(json.loads(LABEL_JSON.read_text()))
lab_txt = parse_txt_labels(LABEL_TXT)
records_expected = [f"x{i:02d}" for i in range(1,36)]
if sorted(lab_json) != records_expected or sorted(lab_txt) != records_expected:
    raise RuntimeError("Official x record universe mismatch")
for r in records_expected:
    if lab_json[r] != lab_txt[r]:
        raise RuntimeError(f"Official JSON/TXT labels disagree for {r}")

Y = np.full(len(UID), -1, np.int8)
REC = np.empty(len(UID), dtype="U8")
EPOCH = np.empty(len(UID), dtype=np.int64)
bad = []
for i, u in enumerate(UID):
    r, e = u.rsplit(":", 1)
    e = int(e)
    REC[i] = r
    EPOCH[i] = e
    if r not in lab_json or e < 0 or e >= len(lab_json[r]):
        bad.append(u)
    else:
        Y[i] = 1 if lab_json[r][e] == "A" else 0

if bad or np.any(Y < 0):
    raise RuntimeError(f"Official label alignment failed: {bad[:10]}")
if sorted(np.unique(REC).tolist()) != records_expected:
    raise RuntimeError("Final prediction record universe is not exact x01-x35")

M_FINAL = metric_row(Y, P_FINAL, THR)
M_PHYS = metric_row(Y, P_PHYS, THR)
M_QML = metric_row(Y, P_QML, THR)

if abs(M_FINAL["accuracy"] - EXPECTED_ACC) > 1e-12:
    raise RuntimeError(f"Promoted final accuracy reproduction failed: {M_FINAL['accuracy']}")
if abs(M_FINAL["accuracy"] - float(saved_report["fusion"]["accuracy"])) > 1e-12:
    raise RuntimeError("Saved final report disagrees with direct Stage11 reproduction")

print("="*110)
print("STAGE 11 PROVENANCE + OFFICIAL LABEL ALIGNMENT: PASS")
print("="*110)
print("Final SHA:", EXPECTED_FUSED_SHA)
print("Exact fusion reconstruction max |Δ|:", RECON_MAX)
print("Official JSON/TXT agreement: PASS")
print("Rows:", len(Y), "Records:", len(np.unique(REC)))
print("Promoted final official-x accuracy:", 100*M_FINAL["accuracy"])
print("No training/tuning/model selection performed.")


In [ ]:

# Cell 4 — 11.1 Standard Evaluation: final metrics + class-wise metrics + final-system table

official_metrics = {
    "schema": "QML_SleepNet_STAGE11_OFFICIAL_METRICS_v1",
    "prediction_sha256": EXPECTED_FUSED_SHA,
    "evaluation_rows": int(len(Y)),
    "evaluation_records": int(len(np.unique(REC))),
    "threshold": THR,
    "final_system": "FIXED_LOGIT_PHYS_25_QML_75",
    "metrics": M_FINAL,
    "scientific_status": freeze["scientific_status"],
}
atomic_json(OUT / "STAGE11_OFFICIAL_METRICS.json", official_metrics)

pred = (P_FINAL >= THR).astype(np.int8)
pr, rc, f1s, support = precision_recall_fscore_support(
    Y, pred, labels=[0,1], zero_division=0
)
class_rows = []
for label, name, p_, r_, f_, s_ in zip(
    [0,1], ["Normal (N)", "Apnea (A)"], pr, rc, f1s, support
):
    class_rows.append({
        "class_label": int(label),
        "class_name": name,
        "precision": float(p_),
        "recall": float(r_),
        "f1": float(f_),
        "support": int(s_),
    })
CLASS_DF = pd.DataFrame(class_rows)
atomic_csv(OUT / "STAGE11_CLASSWISE_METRICS.csv", CLASS_DF)

same_x_rows = [
    {
        "system": "FINAL_QML_INCLUSIVE_PROMOTED",
        "role": "Final metric-first QML-inclusive system; development-promoted before fused x scoring",
        "quantum_in_final_function": True,
        "physiology_logit_weight": W_PHYS,
        "qml_logit_weight": W_QML,
        **M_FINAL,
    },
    {
        "system": "STAGE15A_PHYSIOLOGY_CLASSICAL",
        "role": "Best classical-only historical benchmark",
        "quantum_in_final_function": False,
        "physiology_logit_weight": 1.0,
        "qml_logit_weight": 0.0,
        **M_PHYS,
    },
    {
        "system": "CANONICAL_GUIDE_QML",
        "role": "Canonical guide-primary QML architecture",
        "quantum_in_final_function": True,
        "physiology_logit_weight": 0.0,
        "qml_logit_weight": 1.0,
        **M_QML,
    },
]
SYSTEM_DF = pd.DataFrame(same_x_rows)
atomic_csv(OUT / "STAGE11_FINAL_RESULTS_TABLE.csv", SYSTEM_DF)

display(CLASS_DF)
display(SYSTEM_DF[[
    "system","role","accuracy","balanced_accuracy","sensitivity",
    "specificity","f1","mcc","auroc","auprc"
]])


In [ ]:

# Cell 5 — 11.1 plots: confusion matrix, ROC, PR

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(
    Y, pred, labels=[0,1], display_labels=["Normal","Apnea"],
    values_format="d", ax=ax
)
ax.set_title("Promoted Final QML-SleepNet — Official x Confusion Matrix")
fig.tight_layout()
atomic_savefig(PLOTS / "STAGE11_CONFUSION_MATRIX.png", fig)
plt.show()
plt.close(fig)

fpr, tpr, _ = roc_curve(Y, P_FINAL)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(fpr, tpr, label=f"Final AUROC = {M_FINAL['auroc']:.4f}")
ax.plot([0,1], [0,1], linestyle="--", label="Chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Promoted Final QML-SleepNet — ROC")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
atomic_savefig(PLOTS / "STAGE11_ROC_CURVE.png", fig)
plt.show()
plt.close(fig)

precision_curve, recall_curve, _ = precision_recall_curve(Y, P_FINAL)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(recall_curve, precision_curve, label=f"Final AUPRC = {M_FINAL['auprc']:.4f}")
ax.axhline(float(Y.mean()), linestyle="--", label=f"Apnea prevalence = {Y.mean():.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Promoted Final QML-SleepNet — Precision–Recall")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
atomic_savefig(PLOTS / "STAGE11_PR_CURVE.png", fig)
plt.show()
plt.close(fig)

print("11.1 evaluation plots saved.")


In [ ]:

# Cell 6 — per-record metrics: 35 record clusters are the primary inferential units

SYSTEMS = {
    "FINAL_QML_INCLUSIVE_PROMOTED": P_FINAL,
    "STAGE15A_PHYSIOLOGY_CLASSICAL": P_PHYS,
    "CANONICAL_GUIDE_QML": P_QML,
}

record_rows = []
for r in records_expected:
    idx = np.where(REC == r)[0]
    yr = Y[idx]
    for name, score in SYSTEMS.items():
        mr = metric_row(yr, score[idx], THR)
        record_rows.append({
            "record_name": r,
            "system": name,
            "n_minutes": int(len(idx)),
            "apnea_prevalence": float(np.mean(yr)),
            **mr,
        })

RECORD_DF = pd.DataFrame(record_rows)
atomic_csv(OUT / "STAGE11_PER_RECORD_METRICS.csv", RECORD_DF)

wide_acc = RECORD_DF.pivot(index="record_name", columns="system", values="accuracy").loc[records_expected]
if wide_acc.shape != (35,3) or wide_acc.isna().any().any():
    raise RuntimeError("Per-record accuracy matrix incomplete")

display(RECORD_DF.head(9))

fig, ax = plt.subplots(figsize=(8,5))
data = [
    wide_acc["FINAL_QML_INCLUSIVE_PROMOTED"].to_numpy(),
    wide_acc["STAGE15A_PHYSIOLOGY_CLASSICAL"].to_numpy(),
    wide_acc["CANONICAL_GUIDE_QML"].to_numpy(),
]
ax.boxplot(data, labels=["Final 25/75","Classical","Canonical QML"])
ax.set_ylabel("Per-record accuracy")
ax.set_title("Official x Record-Cluster Accuracy Distribution")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
atomic_savefig(PLOTS / "STAGE11_RECORD_ACCURACY_DISTRIBUTION.png", fig)
plt.show()
plt.close(fig)


In [ ]:

# Cell 7 — 11.2 paired t-tests + paired Cohen's d_z + t confidence intervals + Holm correction

pairs = [
    ("FINAL_QML_INCLUSIVE_PROMOTED", "STAGE15A_PHYSIOLOGY_CLASSICAL"),
    ("FINAL_QML_INCLUSIVE_PROMOTED", "CANONICAL_GUIDE_QML"),
    ("CANONICAL_GUIDE_QML", "STAGE15A_PHYSIOLOGY_CLASSICAL"),
]

stat_rows = []
for a, b in pairs:
    stat_rows.append(
        paired_stats(
            wide_acc[a].to_numpy(),
            wide_acc[b].to_numpy(),
            a, b
        )
    )

STAT_DF = pd.DataFrame(stat_rows)
STAT_DF["p_value_holm"] = holm_adjust(STAT_DF["p_value_raw"].to_numpy())
STAT_DF["significant_holm_0p05"] = STAT_DF["p_value_holm"] < 0.05
STAT_DF["statistical_unit"] = "official x record cluster"
STAT_DF["metric_tested"] = "per-record minute-level accuracy"
STAT_DF["interpretation_boundary"] = (
    "35 record clusters; do not interpret 17,248 minute rows as iid"
)

atomic_csv(OUT / "STAGE11_PAIRED_STATISTICS.csv", STAT_DF)
display(STAT_DF)

print(
    "\nPrimary inferential tests use 35 record clusters. "
    "The p-values are NOT computed by pretending minute epochs are independent."
)


In [ ]:

# Cell 8 — record-cluster bootstrap: final metric CIs + paired metric-delta CIs

idx_by_record = {r: np.where(REC == r)[0] for r in records_expected}
rng = np.random.default_rng(BOOTSTRAP_SEED)

metric_names = [
    "accuracy","balanced_accuracy","precision","sensitivity","specificity",
    "f1","mcc","auroc","auprc","brier","nll"
]
final_draws = {k: np.empty(BOOTSTRAP_REPS, float) for k in metric_names}

pair_defs = [
    ("FINAL_QML_INCLUSIVE_PROMOTED", "STAGE15A_PHYSIOLOGY_CLASSICAL"),
    ("FINAL_QML_INCLUSIVE_PROMOTED", "CANONICAL_GUIDE_QML"),
    ("CANONICAL_GUIDE_QML", "STAGE15A_PHYSIOLOGY_CLASSICAL"),
]
delta_metrics = ["accuracy","balanced_accuracy","f1","mcc","auroc","auprc"]
delta_draws = {
    (a,b,k): np.empty(BOOTSTRAP_REPS, float)
    for a,b in pair_defs for k in delta_metrics
}

for boot in range(BOOTSTRAP_REPS):
    sampled_records = rng.choice(records_expected, size=len(records_expected), replace=True)
    ii = np.concatenate([idx_by_record[r] for r in sampled_records])

    mf = metric_row(Y[ii], P_FINAL[ii], THR)
    for k in metric_names:
        final_draws[k][boot] = mf[k]

    boot_metrics = {
        name: metric_row(Y[ii], p[ii], THR)
        for name, p in SYSTEMS.items()
    }
    for a,b in pair_defs:
        for k in delta_metrics:
            delta_draws[(a,b,k)][boot] = boot_metrics[a][k] - boot_metrics[b][k]

boot_rows = []

for k in metric_names:
    d = final_draws[k]
    boot_rows.append({
        "analysis": "FINAL_METRIC_CI",
        "system_a": "FINAL_QML_INCLUSIVE_PROMOTED",
        "system_b": "",
        "metric": k,
        "observed": M_FINAL[k],
        "bootstrap_mean": float(np.mean(d)),
        "ci95_low": float(np.quantile(d,0.025)),
        "ci95_high": float(np.quantile(d,0.975)),
        "probability_delta_gt_0": np.nan,
        "replicates": BOOTSTRAP_REPS,
        "resampling_unit": "official x record cluster",
    })

observed_system_metrics = {
    "FINAL_QML_INCLUSIVE_PROMOTED": M_FINAL,
    "STAGE15A_PHYSIOLOGY_CLASSICAL": M_PHYS,
    "CANONICAL_GUIDE_QML": M_QML,
}
for a,b in pair_defs:
    for k in delta_metrics:
        d = delta_draws[(a,b,k)]
        obs = observed_system_metrics[a][k] - observed_system_metrics[b][k]
        boot_rows.append({
            "analysis": "PAIRED_METRIC_DELTA_CI",
            "system_a": a,
            "system_b": b,
            "metric": k,
            "observed": float(obs),
            "bootstrap_mean": float(np.mean(d)),
            "ci95_low": float(np.quantile(d,0.025)),
            "ci95_high": float(np.quantile(d,0.975)),
            "probability_delta_gt_0": float(np.mean(d > 0)),
            "replicates": BOOTSTRAP_REPS,
            "resampling_unit": "official x record cluster",
        })

BOOT_DF = pd.DataFrame(boot_rows)
atomic_csv(OUT / "STAGE11_RECORD_CLUSTER_BOOTSTRAP_CI.csv", BOOT_DF)

display(BOOT_DF.head(15))


In [ ]:

# Cell 9 — 11.3 VQC vs QSVC: matched development-only four-fold evidence

e2_compare = pd.read_csv(E2_COMPARE)
qsvc_raw = pd.read_csv(E2_QSVC_FOLDS)
vqc_raw = pd.read_csv(E2_VQC_FOLDS)

if e2_decision.get("outer_heldout_labels_accessed") is not False:
    raise RuntimeError("Stage15E2 comparison provenance says outer held-out labels were accessed")
if e2_decision.get("stage15d_ab_modified") is not False:
    raise RuntimeError("Stage15E2 unexpectedly modified Stage15D A+B")
if e2_decision.get("qubits") != 6 or e2_decision.get("reduction") != "PCA6":
    raise RuntimeError("Stage15E2 guide-parity QML identity drift")
if e2_decision.get("best_quantum_branch") != "ANISOTROPIC_ANGLE_QSVC":
    raise RuntimeError("Unexpected Stage15E2 selected quantum branch")

# Selected fixed QSVC is rank 0 in each of the four real inner folds.
qsvc = qsvc_raw.loc[qsvc_raw["rank_id"] == 0].copy().sort_values("fold")
vqc = vqc_raw.copy().sort_values("fold")
if qsvc["fold"].tolist() != [0,1,2,3] or vqc["fold"].tolist() != [0,1,2,3]:
    raise RuntimeError("VQC/QSVC four-fold alignment failure")

dev_cmp = qsvc[["fold","auroc","auprc","balanced_accuracy","group_macro_auroc"]].copy()
dev_cmp = dev_cmp.rename(columns={c:f"qsvc_{c}" for c in dev_cmp.columns if c!="fold"})
for c in ["auroc","auprc","balanced_accuracy","group_macro_auroc"]:
    dev_cmp[f"vqc_{c}"] = vqc[c].to_numpy()

metric_map = ["auroc","auprc","balanced_accuracy","group_macro_auroc"]
dev_stat_rows = []
for k in metric_map:
    a = dev_cmp[f"qsvc_{k}"].to_numpy()
    b = dev_cmp[f"vqc_{k}"].to_numpy()
    row = paired_stats(a, b, "ANISOTROPIC_ANGLE_QSVC", "VQC_REALAMPLITUDES_SPSA")
    row["metric_tested"] = k
    row["statistical_unit"] = "four matched strict development folds"
    row["interpretation_boundary"] = "EXPLORATORY / LOW POWER: n=4 development folds; not official-x inference"
    dev_stat_rows.append(row)

DEV_STAT = pd.DataFrame(dev_stat_rows)
DEV_STAT["p_value_holm"] = holm_adjust(DEV_STAT["p_value_raw"].to_numpy())
DEV_STAT["significant_holm_0p05"] = DEV_STAT["p_value_holm"] < 0.05

atomic_csv(OUT / "STAGE11_VQC_QSVC_DEVELOPMENT_COMPARISON.csv", dev_cmp)
atomic_csv(OUT / "STAGE11_VQC_QSVC_DEVELOPMENT_STATISTICS.csv", DEV_STAT)

display(e2_compare)
display(dev_cmp)
display(DEV_STAT[[
    "metric_tested","n_pairs","mean_paired_difference","cohen_dz",
    "p_value_raw","p_value_holm","significant_holm_0p05",
    "interpretation_boundary"
]])


In [ ]:

# Cell 10 — 11.3 Quantum vs Classical + ablation, with evidence domains kept separate

# Development-domain matched-representation summary.
quantum_classical_rows = []
for _, r in e2_compare.iterrows():
    quantum_classical_rows.append({
        "evidence_domain": "Stage15E2 strict development folds; 6-D matched representation",
        "system": r["branch"],
        "mean_auroc": float(r["mean_auroc"]),
        "worst_fold_auroc": float(r["worst_fold_auroc"]),
        "mean_auprc": float(r["mean_auprc"]),
        "mean_group_macro_auroc": float(r["mean_group_macro_auroc"]),
        "quantum_advantage_claim_allowed": False,
    })

# Official-x same-row system comparison; still NOT a quantum-advantage experiment.
for name, m, role in [
    ("FINAL_QML_INCLUSIVE_PROMOTED", M_FINAL, "25% classical + 75% QML final"),
    ("STAGE15A_PHYSIOLOGY_CLASSICAL", M_PHYS, "classical-only benchmark"),
    ("CANONICAL_GUIDE_QML", M_QML, "guide-primary QML system"),
]:
    quantum_classical_rows.append({
        "evidence_domain": "Official x same 17,248 frozen project rows",
        "system": name,
        "mean_auroc": float(m["auroc"]),
        "worst_fold_auroc": np.nan,
        "mean_auprc": float(m["auprc"]),
        "mean_group_macro_auroc": np.nan,
        "quantum_advantage_claim_allowed": False,
        "role": role,
    })

QC_DF = pd.DataFrame(quantum_classical_rows)
atomic_csv(OUT / "STAGE11_QUANTUM_CLASSICAL_COMPARISON.csv", QC_DF)

# Use only genuine parent-only ablations; omit neutralization duplicates from hard-class comparison.
abl = pd.read_csv(S10_ABLATION)
keep = [
    "PROMOTED_FINAL_PHYS25_QML75",
    "PHYSIOLOGY_PARENT_ONLY",
    "QML_PARENT_ONLY",
]
ABL_DF = abl.loc[abl["system"].isin(keep)].copy()
if set(ABL_DF["system"]) != set(keep):
    raise RuntimeError("Expected promoted parent-ablation rows missing")
ABL_DF["interpretation"] = ABL_DF["system"].map({
    "PROMOTED_FINAL_PHYS25_QML75": "locked promoted final",
    "PHYSIOLOGY_PARENT_ONLY": "remove QML fusion contribution; classical parent only",
    "QML_PARENT_ONLY": "remove Stage15A physiology parent; canonical QML parent only",
})
ABL_DF["promotion_allowed_from_stage11"] = False
atomic_csv(OUT / "STAGE11_ABLATION_COMPARISON.csv", ABL_DF)

display(QC_DF)
display(ABL_DF[[
    "system","accuracy","balanced_accuracy","f1","mcc","auroc","auprc",
    "accuracy_delta_vs_promoted_final_pp","interpretation"
]])


In [ ]:

# Cell 11 — 11.1 original-Challenge / state-of-the-art context with explicit non-identical protocol warning

# Authoritative historical Event-2 leaderboard from PhysioNet Challenge 2000.
# Source: https://physionet.org/content/challenge-2000/1.0.0/
challenge_rows = [
    {
        "entry": "McNames / Fraser / Rechtsteiner",
        "context": "PhysioNet/CinC Challenge 2000 Event 2",
        "correct_minutes": 15994,
        "denominator_minutes": 17268,
        "accuracy": 15994/17268,
        "source_url": "https://physionet.org/content/challenge-2000/1.0.0/",
        "same_x01_x35_task_family": True,
        "protocol_identical_to_current_project": False,
    },
    {
        "entry": "Raymond / Cayton / Bates / Chappell",
        "context": "PhysioNet/CinC Challenge 2000 Event 2",
        "correct_minutes": 15939,
        "denominator_minutes": 17268,
        "accuracy": 15939/17268,
        "source_url": "https://physionet.org/content/challenge-2000/1.0.0/",
        "same_x01_x35_task_family": True,
        "protocol_identical_to_current_project": False,
    },
    {
        "entry": "de Chazal et al.",
        "context": "PhysioNet/CinC Challenge 2000 Event 2",
        "correct_minutes": 15432,
        "denominator_minutes": 17268,
        "accuracy": 15432/17268,
        "source_url": "https://physionet.org/content/challenge-2000/1.0.0/",
        "same_x01_x35_task_family": True,
        "protocol_identical_to_current_project": False,
    },
    {
        "entry": "QML-SleepNet promoted final",
        "context": "Current frozen project evaluation",
        "correct_minutes": int(M_FINAL["tn"] + M_FINAL["tp"]),
        "denominator_minutes": int(M_FINAL["n"]),
        "accuracy": float(M_FINAL["accuracy"]),
        "source_url": "internal frozen Stage11 reproduction",
        "same_x01_x35_task_family": True,
        "protocol_identical_to_current_project": True,
    },
]
EXT_DF = pd.DataFrame(challenge_rows)
atomic_csv(OUT / "STAGE11_EXTERNAL_BENCHMARK_CONTEXT.csv", EXT_DF)

ext_limit = {
    "authoritative_reference": "PhysioNet/Computing in Cardiology Challenge 2000 Event 2",
    "reference_url": "https://physionet.org/content/challenge-2000/1.0.0/",
    "challenge_reference_minutes": 17268,
    "current_project_scored_minutes": int(M_FINAL["n"]),
    "minute_count_difference": int(17268-M_FINAL["n"]),
    "current_project_policy": "complete observed raw 60-second ECG minute AND available official annotation; no terminal partial-minute fabrication/padding",
    "direct_sota_claim_allowed": False,
    "reason": (
        "The official historical Challenge leaderboard uses 17,268 reference-annotated minutes, "
        "whereas the current frozen project chain evaluates 17,248 complete observed 60-s epochs. "
        "The record/task family is aligned but coverage is not identical."
    ),
    "modern_sota_claim": "NOT ASSERTED: no modern paper is called protocol-matched SOTA unless its exact held-out x01-x35 coverage and evaluation contract are independently verified.",
}
atomic_json(OUT / "STAGE11_EXTERNAL_BENCHMARK_LIMITATIONS.json", ext_limit)

display(EXT_DF)
print(json.dumps(ext_limit, indent=2))


In [ ]:

# Cell 12 — 11.4 consolidated plots + findings + limitations + robustness/XAI summary

# Same-row final-system comparison plot.
plot_df = SYSTEM_DF.copy()
plot_df["display_name"] = plot_df["system"].map({
    "FINAL_QML_INCLUSIVE_PROMOTED": "Final QML-inclusive",
    "STAGE15A_PHYSIOLOGY_CLASSICAL": "Classical benchmark",
    "CANONICAL_GUIDE_QML": "Canonical QML",
})
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(plot_df["display_name"], 100*plot_df["accuracy"])
ax.set_ylabel("Official-x accuracy (%)")
ax.set_title("Same-Row Official-x System Comparison")
ax.set_ylim(max(0, 100*plot_df["accuracy"].min()-2), min(100, 100*plot_df["accuracy"].max()+1))
for i, v in enumerate(100*plot_df["accuracy"]):
    ax.text(i, v+0.05, f"{v:.3f}%", ha="center", va="bottom")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
atomic_savefig(PLOTS / "STAGE11_FINAL_COMPARISON.png", fig)
plt.show()
plt.close(fig)

pert = pd.read_csv(S10_PERTURB)
scope_items = json.loads(S10_SCOPE.read_text())

summary = {
    "final_designation": {
        "system": "FIXED_LOGIT_PHYS_25_QML_75",
        "prediction_sha256": EXPECTED_FUSED_SHA,
        "official_x_accuracy": M_FINAL["accuracy"],
        "designation_basis": "already promoted on strict development OOF before fused x scoring; Stage11 does not reselect",
    },
    "same_row_context": {
        "best_classical_accuracy": M_PHYS["accuracy"],
        "canonical_qml_accuracy": M_QML["accuracy"],
        "final_minus_classical_accuracy_pp": 100*(M_FINAL["accuracy"]-M_PHYS["accuracy"]),
        "final_minus_canonical_qml_accuracy_pp": 100*(M_FINAL["accuracy"]-M_QML["accuracy"]),
    },
    "xai": {
        "status": "COMPLETE_FOR_PROMOTED_FINAL",
        "manifest": str(XAI_MANIFEST),
        "exact_top_level_method": xai_manifest["exact_top_level_explanation"]["method"],
        "weights": xai_manifest["exact_top_level_explanation"]["weights"],
    },
    "robustness": {
        "status": "COMPLETE_WITH_SCOPE_LIMITATIONS",
        "manifest": str(S10_MANIFEST),
        "perturbation_artifact": str(S10_PERTURB),
        "scope_limitation": s10_manifest["stage10_2_noise_and_temporal_coverage"]["scope_limitation"],
    },
    "key_findings": [
        "Promoted final QML-inclusive system reproduces 90.8627% accuracy on the frozen 17,248-row project official-x universe.",
        "The final system improves the canonical guide-primary QML parent by approximately 0.748 percentage points accuracy on the same rows.",
        "The classical Stage15A parent remains approximately 0.157 percentage points higher in raw official-x accuracy than the promoted QML-inclusive final system.",
        "Stage15E2 guide-parity development evidence places QSVC slightly above VQC and the matched 6-D classical proxy in mean AUROC, but no quantum-advantage claim is made.",
        "Promoted robustness results show small changes under inherited Stage06-channel perturbations, but these are not full dual-parent raw-ECG perturbation tests.",
    ],
    "limitations": [
        freeze["scientific_status"],
        "Official-x inferential statistics use 35 record clusters; minute rows are serially correlated and are not treated as iid.",
        "Record clusters are not automatically claimed to be 35 unique subjects.",
        "Historical Challenge Event-2 denominator is 17,268 while this project scores 17,248 complete observed 60-s epochs; historical leaderboard comparison is contextual only.",
        "Stage10 noise/temporal perturbation modifies the Stage06 temporal branch while the promoted physiology parent remains clean.",
        "No sleep-stage target exists in the frozen current artifact contract.",
        "The current project does not possess OSA/CSA/Mixed/Hypopnea ground truth required by the roadmap's literal five-class Stage08 formulation.",
        "No quantum-advantage claim is supported.",
    ],
}
atomic_json(OUT / "STAGE11_KEY_FINDINGS_LIMITATIONS.json", summary)

display(pert)
print(json.dumps(summary, indent=2))


In [ ]:

# Cell 13 — Stage11 guide-contract closure

guide_contract = {
    "roadmap_source": "QML-SleepNet Research Roadmap.png",
    "stage": "Stage 11 — Final Evaluation Protocol (As per Original Paper Style)",
    "11.1_standard_evaluation": {
        "status": "COMPLETE",
        "official_test_set": "x01-x35, frozen project-valid 17,248 complete observed 60-s epochs",
        "classwise_metrics": "STAGE11_CLASSWISE_METRICS.csv",
        "plots": [
            "plots/STAGE11_CONFUSION_MATRIX.png",
            "plots/STAGE11_ROC_CURVE.png",
            "plots/STAGE11_PR_CURVE.png",
        ],
        "state_of_art_context": "STAGE11_EXTERNAL_BENCHMARK_CONTEXT.csv",
        "fairness_boundary": "historical Challenge uses 17,268 minutes; comparison is contextual, not protocol-identical",
    },
    "11.2_statistical_analysis": {
        "status": "COMPLETE",
        "paired_t_tests": "STAGE11_PAIRED_STATISTICS.csv",
        "cohen_d": "paired Cohen d_z on 35 record-level accuracy differences",
        "confidence_intervals": "t-CI for paired mean record accuracy differences + 5000-replicate record-cluster bootstrap",
        "bootstrap": "STAGE11_RECORD_CLUSTER_BOOTSTRAP_CI.csv",
        "multiple_testing": "Holm correction across primary official-x paired tests",
        "primary_statistical_unit": "35 official x record clusters",
    },
    "11.3_comparison": {
        "status": "COMPLETE",
        "vqc_vs_qsvc": [
            "STAGE11_VQC_QSVC_DEVELOPMENT_COMPARISON.csv",
            "STAGE11_VQC_QSVC_DEVELOPMENT_STATISTICS.csv",
        ],
        "quantum_vs_classical": "STAGE11_QUANTUM_CLASSICAL_COMPARISON.csv",
        "ablation": "STAGE11_ABLATION_COMPARISON.csv",
        "evidence_domain_rule": "development-fold QML comparisons and official-x system comparisons are kept separate",
        "quantum_advantage_claim_allowed": False,
    },
    "11.4_final_results": {
        "status": "COMPLETE",
        "best_model_selection": "FINAL_DESIGNATION_ONLY_NO_X_DRIVEN_SELECTION",
        "final_designated_system": "FIXED_LOGIT_PHYS_25_QML_75",
        "final_prediction_sha256": EXPECTED_FUSED_SHA,
        "summary_table": "STAGE11_FINAL_RESULTS_TABLE.csv",
        "summary_plot": "plots/STAGE11_FINAL_COMPARISON.png",
        "findings_limitations": "STAGE11_KEY_FINDINGS_LIMITATIONS.json",
    },
    "footer_requirements": {
        "original_paper_results": "authoritative historical PhysioNet Challenge Event-2 leaderboard included contextually",
        "same_dataset_and_protocol": "same Apnea-ECG/x01-x35 task family; exact minute coverage difference is explicitly disclosed",
        "consistent_metrics": True,
        "fair_comparison": True,
    },
    "training_performed": False,
    "tuning_performed": False,
    "model_selection_performed_in_stage11": False,
    "next": "Stage 12 — Documentation and Reproducibility",
}
atomic_json(OUT / "STAGE11_GUIDE_CONTRACT.json", guide_contract)

print(json.dumps(guide_contract, indent=2))


In [ ]:

# Cell 14 — artifact hash inventory + authoritative Stage11 final manifest

# Hash critical inputs.
input_paths = {
    "promoted_final_prediction": FUSED_X,
    "promoted_final_freeze": FUSION_FREEZE,
    "promoted_final_x_report": FUSION_REPORT,
    "promoted_final_development_decision": FUSION_DECISION,
    "physiology_parent_prediction": PHYS_X,
    "qml_parent_prediction": QML_X,
    "official_label_json": LABEL_JSON,
    "official_label_txt": LABEL_TXT,
    "promoted_xai_manifest": XAI_MANIFEST,
    "promoted_stage10_manifest": S10_MANIFEST,
    "stage15e2_qml_decision": E2_DECISION,
    "stage15e2_qml_comparison": E2_COMPARE,
}

hash_rows = []
for name, path in input_paths.items():
    hash_rows.append({
        "kind": "input",
        "artifact": name,
        "path": str(path),
        "sha256": sha256_file(path),
    })

# Hash Stage11 outputs generated so far (manifest/hash inventory themselves excluded to avoid circularity).
for path in sorted(OUT.rglob("*")):
    if not path.is_file():
        continue
    if path.name in {"STAGE11_ARTIFACT_HASHES.csv", "STAGE11_FINAL_MANIFEST.json"}:
        continue
    hash_rows.append({
        "kind": "stage11_output",
        "artifact": path.name,
        "path": str(path),
        "sha256": sha256_file(path),
    })

HASH_DF = pd.DataFrame(hash_rows)
atomic_csv(OUT / "STAGE11_ARTIFACT_HASHES.csv", HASH_DF)

final_manifest = {
    "schema": "QML_SleepNet_STAGE11_FINAL_EVALUATION_90P8627_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "roadmap_stage": "Stage 11 — Final Evaluation Protocol",
    "roadmap_source": "QML-SleepNet Research Roadmap.png",
    "final_designated_system": {
        "name": "FIXED_LOGIT_PHYS_25_QML_75",
        "description": "25% Stage15A physiology + 75% guide-primary QML, fixed logit fusion",
        "prediction_sha256": EXPECTED_FUSED_SHA,
        "weights": {"physiology": W_PHYS, "qml": W_QML},
        "threshold": THR,
        "official_x_rows": int(len(Y)),
        "official_x_records": int(len(np.unique(REC))),
        "metrics": M_FINAL,
    },
    "comparison_roles": {
        "best_classical_only_benchmark": {
            "name": "Stage15A physiology CatBoost + temperature + HMM",
            "accuracy": M_PHYS["accuracy"],
        },
        "canonical_guide_qml": {
            "name": "Bridge + QT Angle-Rx + corrected Stage06 fixed ensemble",
            "accuracy": M_QML["accuracy"],
        },
    },
    "model_development": {
        "training_performed": False,
        "fine_tuning_performed": False,
        "threshold_search_performed": False,
        "fusion_weight_search_performed": False,
        "calibration_refit_performed": False,
        "hmm_refit_performed": False,
        "model_selection_performed_in_stage11": False,
        "best_model_roadmap_interpretation": "final designation/reporting only; no x-driven re-selection",
    },
    "statistics": {
        "primary_unit": "35 official x record clusters",
        "minute_rows_treated_as_iid": False,
        "paired_t_tests": True,
        "paired_cohen_dz": True,
        "holm_multiple_test_correction": True,
        "record_cluster_bootstrap_replicates": BOOTSTRAP_REPS,
        "record_cluster_bootstrap_seed": BOOTSTRAP_SEED,
    },
    "evidence_domains": {
        "official_x_same_row_system_comparison": "17,248 frozen project rows",
        "vqc_vs_qsvc": "matched strict development folds only; exploratory n=4 fold statistics",
        "external_historical_challenge": "context only; 17,268-minute denominator differs",
    },
    "scientific_caveats": [
        freeze["scientific_status"],
        "Final 90.8627% fused x result is not presented as pristine never-seen external validation.",
        "Best classical-only benchmark (91.0192%) is slightly higher in raw x accuracy than the final QML-inclusive system.",
        "No quantum-advantage claim.",
        "Historical Challenge comparison is not protocol-identical because current project scores 17,248 vs official Challenge 17,268 minutes.",
        "Stage10 promoted robustness perturbation is Stage06/QML-channel stress with physiology parent held clean, not full dual-parent end-to-end raw-ECG perturbation.",
    ],
    "guide_contract": "STAGE11_GUIDE_CONTRACT.json",
    "artifact_hashes": "STAGE11_ARTIFACT_HASHES.csv",
    "status": "STAGE11_COMPLETE",
    "next": "Stage 12 — Documentation and Reproducibility",
}
atomic_json(OUT / "STAGE11_FINAL_MANIFEST.json", final_manifest)

print("="*118)
print("QML-SLEEPNET STAGE 11 FINAL EVALUATION PROTOCOL — COMPLETE")
print("="*118)
print("Final designated model:", final_manifest["final_designated_system"]["name"])
print("Final SHA:", EXPECTED_FUSED_SHA)
print("Official-x accuracy:", 100*M_FINAL["accuracy"])
print("Best classical-only benchmark accuracy:", 100*M_PHYS["accuracy"])
print("Canonical guide-QML accuracy:", 100*M_QML["accuracy"])
print("Training performed: NO")
print("Tuning performed: NO")
print("Stage11 x-driven model selection: NO")
print("Quantum-advantage claim: NO")
print("NEXT: Stage 12 — Documentation and Reproducibility")
print("Evidence:", OUT)



## Acceptance rule

Stage 11 is closed only if the final cell prints:

- `QML-SLEEPNET STAGE 11 FINAL EVALUATION PROTOCOL — COMPLETE`
- final SHA `063a017e...`
- official-x accuracy `90.86270871985158`
- training `NO`
- tuning `NO`
- Stage11 x-driven model selection `NO`
- quantum-advantage claim `NO`
- `NEXT: Stage 12 — Documentation and Reproducibility`

If any fail-closed check raises an error, **do not modify a metric, threshold, fusion weight, test label, or model to make the notebook pass**. Return the traceback and affected artifact for audit.

After Stage 11 passes, model/evaluation development remains permanently closed. Stage 12 is documentation and reproducibility only.
